# Testing rater selection (important text tokens) on SmolVLM2

Runs the **`rater_selection`** step — "sample only important text tokens" — on the
real **`HuggingFaceTB/SmolVLM2-2.2B-Instruct`** attention map, and checks the
invariants from `tests/test_rater_selection.py`:

1. importance is a per-text-token **column vector** (rows = #text tokens, col = 1)
2. thresholding **reduces** the count (#kept < #content)
3. every rater is a content token   4. no structural/special token is selected
5. exact top-k count   6. importance is a valid distribution
7. kept = top-#kept by importance   8. always ≥ 1 rater
9. pct is monotonic   10. the gaze vision-weight hook is valid

Pipeline: SmolVLM raw text→vision attention (from the probe) → slice to
`[H, L_t, L_v]` → `select_important_text_tokens` → run the checks.

> **Runtime:** GPU runtime. SmolVLM2 is open (no token needed).

## 1. Install + clone

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words pytest

In [ ]:
!rm -rf text_vision_attention_map          # fresh checkout
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git
%cd text_vision_attention_map

## 2. Build the SmolVLM probe and the rater case
Probe the model once (raw text→vision attention), slice to the text→vision block,
then select the important text tokens.

In [ ]:
import importlib.util, os

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S  = _load("probe_smolvlm", "tests/probe_smolvlm.py")
R  = _load("test_rater_selection", "tests/test_rater_selection.py")
import rater_selection as RS

o = S.make_smolvlm_output()          # raw text->vision attention for a (image, prompt)
assert o is not None, "SmolVLM probe failed to load — see printed reason above."

# tokenizer (cheap: no model weights) to label text rows + detect special tokens
from transformers import AutoProcessor
tokenizer = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-2.2B-Instruct").tokenizer

# slice full [H, L, L] -> text->vision {layer: [H, L_t, L_v]}, and label text rows
maps_per_head, tpos, vpos = RS.sliced_maps_from_full(
    o.raw_scores, o.image_token_mask, o.text_token_mask)
text_tokens = tokenizer.convert_ids_to_tokens(o.input_ids[tpos].tolist())

# scope raters to the user's QUESTION span (drops User:/Assistant:/template)
print("question:", repr(o.question))
case = R.make_case(maps_per_head, text_tokens, pct=0.5, tokenizer=tokenizer,
                   question=o.question, name="smolvlm")
res = case.res
print("text tokens (L_t)      :", len(text_tokens))
print("content tokens (L_t')  :", res.n_content, "  <- only question-span content now")
print("kept raters (#kept)    :", res.n_kept, " | band (layers):", res.band, " | pct:", case.pct)
print("kept tokens            :", res.kept_tokens(text_tokens))

## 3. Run ALL invariant checks on the real SmolVLM rater case

In [ ]:
passed = 0
for fn in R.ALL_CHECKS:
    try:
        fn(case)
        print(f"PASS  {fn.__name__}")
        passed += 1
    except AssertionError as e:
        print(f"FAIL  {fn.__name__}: {e}")

print(f"\n{passed}/{len(R.ALL_CHECKS)} rater-selection checks passed on SmolVLM2.")

## 4. Your two test cases — concrete numbers
(1) importance shape, (2) thresholding reduces the count.

In [ ]:
# (1) importance is a per-text-token column vector
print("importance shape       :", tuple(res.importance.shape),
      "-> as column:", tuple(res.importance.view(-1, 1).shape))
print("rows == #text tokens   :", res.importance.shape[0] == len(text_tokens),
      f"({res.importance.shape[0]} == {len(text_tokens)})")

# (2) thresholding reduces the number of rows
print("before threshold (content):", res.n_content)
print("after  threshold (kept)   :", res.n_kept)
print("reduced?                  :", res.n_kept < res.n_content)

## 5. Qualitative: which text tokens were kept vs dropped
The kept tokens should be the visually-grounded content words; structural tokens
(`<|...|>`, newlines) and low-relevance words are dropped.

In [ ]:
order = sorted(range(len(text_tokens)), key=lambda i: -res.importance[i].item())
print(f"{'token':<18}{'importance':>12}   status")
print('-' * 48)
for i in order:
    if res.rater_mask[i]:
        status = 'KEPT (rater)'
    elif res.content_mask[i]:
        status = 'dropped'
    elif RS.is_punctuation_token(text_tokens[i]):
        status = 'suppressed (punctuation)'
    else:
        status = 'suppressed (structural)'
    print(f"{repr(text_tokens[i]):<18}{res.importance[i].item():>12.4f}   {status}")

## 6. (Optional) sweep pct to see the count shrink

In [ ]:
for pct in (0.0, 0.25, 0.5, 0.75, 0.9):
    r = RS.select_important_text_tokens(
        maps_per_head, text_tokens=text_tokens, tokenizer=tokenizer,
        question=o.question, pct=pct)
    print(f"pct={pct:<4}  kept={r.n_kept:>2}  ->  {r.kept_tokens(text_tokens)}")